In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Read all the JSON files from Bronze layer

df_bronze = spark.read.option("multiline", "true").json("Files/bronze/*.json")

print(f"Records loaded: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.show(3, truncate = False)

StatementMeta(, 1e42b5c2-f304-4f1e-8742-b378e3eb60cf, 4, Finished, Available, Finished, False)

Records loaded: 20
root
 |-- address: string (nullable = true)
 |-- aml_screening_status: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- document_number: string (nullable = true)
 |-- document_type: string (nullable = true)
 |-- employer: string (nullable = true)
 |-- expiry_date: string (nullable = true)
 |-- extraction_confidence: double (nullable = true)
 |-- full_name: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- kyc_risk_band: string (nullable = true)
 |-- last_review_date: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- upload_timestamp: string (nullable = true)

+----------------------------------------------+--------------------+------------+-------------+------------------+--------------+----------------------+-----------+---------------------+--------------------+----------+-------------+----------------+------

In [8]:
from datetime import date

## Add pipeline metadata column

df_silver = df_bronze.withColumn('ingestion_date', F.current_date()) \
.withColumn('pipeline_name', F.lit('bronze_to_silver_kyc')) \
.withColumn('expiry_date', F.to_date('expiry_date', 'yyyy-MM-dd')) \
.withColumn('issue_date', F.to_date('issue_date', 'yyyy-MM-dd')) \
.withColumn('date_of_birth', F.to_date('date_of_birth', 'yyyy-MM-dd')) \
.withColumn('days_until_expiry', F.datediff(F.col('expiry_date'), F.current_date())) \
.withColumn('isexpired', F.when(F.col('days_until_expiry')<0, True).otherwise(False)) \
.withColumn('expiry_status', \
F.when(F.col('days_until_expiry')<0, 'EXPIRED') \
.when((F.col('days_until_expiry')<90) & (F.col('days_until_expiry')>0), 'EXPIRING SOON') \
.otherwise('VALID'))

df_silver.select('customer_id', 'full_name', 'expiry_status', 'days_until_expiry').show(20)

StatementMeta(, 1e42b5c2-f304-4f1e-8742-b378e3eb60cf, 10, Finished, Available, Finished, False)

+------------+--------------------+-------------+-----------------+
| customer_id|           full_name|expiry_status|days_until_expiry|
+------------+--------------------+-------------+-----------------+
|UAE-2024-015|Aisha Binte Abdullah|      EXPIRED|             -370|
|UAE-2024-006|    Maryam Al Nuaimi|      EXPIRED|             -430|
|UAE-2024-014|       Carlos Mendes|      EXPIRED|             -220|
|UAE-2024-009|          Priya Nair|      EXPIRED|             -500|
|UAE-2024-007|     Sultan Al Kaabi|      EXPIRED|             -460|
|UAE-2024-002|   Fatima Al Rashidi|EXPIRING SOON|               50|
|UAE-2024-018|      David Thompson|      EXPIRED|             -469|
|UAE-2024-012|             Liu Wei|        VALID|              130|
|UAE-2024-016|Omar Hassan Al Farsi|      EXPIRED|             -415|
|UAE-2024-020|    Tariq Al Blooshi|      EXPIRED|             -520|
|UAE-2024-005|  Khalid Al Mazrouei|      EXPIRED|             -400|
|UAE-2024-013|       Elena Petrova|      EXPIRED

In [9]:
## write to delta table in silver layer
## delta = Versioned, ACID compliant 

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("kyc_silver")

print("Delta table kyc_silver created successfully")
spark.sql('select count(*) as total_records from kyc_silver').show(20)

StatementMeta(, 1e42b5c2-f304-4f1e-8742-b378e3eb60cf, 11, Finished, Available, Finished, False)

Delta table kyc_silver created successfully
+-------------+
|total_records|
+-------------+
|           20|
+-------------+

